# Reinforcement Learning

# 4. Parametric Bandits

The objective of this lab is to recommend contents (here movies) using **parametric bandits**. The rewards are binary (like or dislike).


## Imports

In [2]:
import numpy as np
import pandas as pd

In [3]:
from ipywidgets import AppLayout, Button, GridspecLayout, Image, Layout

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer

## Data

We work on a catalogue of 1037 movies available in 2015.

In [5]:
catalogue = pd.read_pickle('movie_database.pickle')

In [6]:
len(catalogue)

1037

In [7]:
catalogue.head()

,Actors,Awards,Country,Director,Genre,Language,Rated,Released,Title,imdbID,imdbRating,Metascore,Box_office,imdbVotes,Runtime,poster
0,"[Mark Hamill, Harrison Ford, Carrie Fisher, Bi...",Won 1 Oscar. Another 15 wins & 18 nominations.,[USA],[Irvin Kershner],"[Action, Adventure, Fantasy]",[English],[PG],1980-06-20,Star Wars: Episode V - The Empire Strikes Back,tt0080684,8.8,79.0,290158751.0,799579.0,124.0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
1,"[Kareem Abdul-Jabbar, Lloyd Bridges, Peter Gra...",Nominated for 1 Golden Globe. Another 2 wins &...,[USA],"[Jim Abrahams, David Zucker, Jerry Zucker]",[Comedy],[English],[PG],1980-07-02,Airplane!,tt0080339,7.8,NaN,83400000.0,154994.0,88.0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
2,"[John Belushi, Dan Aykroyd, James Brown, Cab C...",1 win.,[USA],[John Landis],"[Action, Comedy, Crime]",[English],[R],1980-06-20,The Blues Brothers,tt0080455,7.9,NaN,54200000.0,138196.0,133.0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
3,"[Jack Nicholson, Shelley Duvall, Danny Lloyd, ...",3 wins & 5 nominations.,"[USA, UK]",[Stanley Kubrick],"[Drama, Horror]",[English],[R],1980-05-23,The Shining,tt0081505,8.4,61.0,NaN,584323.0,146.0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...
4,"[Anthony Hopkins, John Hurt, Anne Bancroft, Jo...",Nominated for 8 Oscars. Another 10 wins & 14 n...,"[USA, UK]",[David Lynch],"[Biography, Drama]",[English],[PG],1980-10-10,The Elephant Man,tt0080678,8.2,NaN,NaN,156572.0,124.0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00...


The features are the following:

|Column|Description|Type|
|:---|:---|:---|
|Actors| Actors staring | list of strings|
|Awards| Awards received| string|
|Country| Country of origin| list of strings|
|Director| Director(s) of the movie|  list of strings|
|Genre| Genres (Action, ...) | list of strings|
|Language| Language(s) spoken |list of strings|
|Rated| Public rating (G = General, R = Restricted, ...)| list of strings|
|Released| Date of the movie| date|
|Title|Title of the movie|string|
|imdbID| IMDB id| string|
|imdbRating| IMDB rating (between 0 and 10)| float|
|Metascore| Metacritic score (between 0 and 100)|float|
|Box_office| Total money generated|float|
|imdbVotes| Number of IMDB votes| float|
|Runtime| Duration of the movie (in minutes)|float|
|poster| Poster of the movie (jpg)| binary string|

In [8]:
# Display the posters

def get_poster(k, scale=1):
    return Image(
        value = catalogue.loc[k].poster,
        format = 'jpg',
        width = 130 * scale,
        height = 200 * scale,
    )

def display_posters(index=None, n_col=5, n_rows=4):
    """Display posters in the order given by the index (if any)."""
    if index is None:
        index = np.arange(len(catalogue))
    if len(index):
        n_rows = min(n_rows, int(np.ceil(len(index) / n_col)))
        grid = GridspecLayout(n_rows, n_col)
        k = 0
        for i in range(n_rows):
            for j in range(n_col):
                if k < len(index):
                    grid[i, j] = get_poster(index[k])
                k += 1 
        return grid

In [9]:
display_posters()

GridspecLayout(children=(Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xf…

## Features

We will describe each movie by some features, for instance its genre.

In [10]:
mlb = MultiLabelBinarizer()

In [11]:
movies = pd.DataFrame(mlb.fit_transform(catalogue['Genre']), columns=mlb.classes_)

In [12]:
movies.head()

,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
movies.columns

Index(['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror',
       'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sport', 'Thriller',
       'War', 'Western'],
      dtype='object')

## User

Each user will be modeled by a vector of weights (positive or negative) on each feature. 

In [14]:
user = pd.DataFrame(0, index = [0], columns=movies.columns)
user['Action'] = 2
user['Crime'] = 1
user['Sci-Fi'] = -2

## To do 1

* Display the favorite movies of this user.
* Test another user, and quantify their similarity (e.g., proportion of common top-100 movies).

In [15]:
user_1 = user.iloc[0].reindex(movies.columns).astype(float)
user_2 = pd.Series(0, index=movies.columns, dtype=float)
user_2['Action'] = 2
user_2['Thriller'] = 1

user_1_ranking = catalogue.loc[:, ['Title', 'Genre']].copy()
user_1_ranking['score'] = movies @ user_1
user_1_ranking = user_1_ranking.sort_values(['score', 'Title'], ascending=[False, True])

user_2_ranking = catalogue.loc[:, ['Title', 'Genre']].copy()
user_2_ranking['score'] = movies @ user_2
user_2_ranking = user_2_ranking.sort_values(['score', 'Title'], ascending=[False, True])

display(user_1_ranking.head(10))
display_posters(user_1_ranking.head(10).index.to_numpy())

display(user_2_ranking.head(10))
common_top_100 = len(set(user_1_ranking.head(100).index) & set(user_2_ranking.head(100).index)) / 100
print("Second user weights:", {'Action': 2, 'Thriller': 1})
print(f"Common movies in the two top-100 lists: {common_top_100:.0%}")

,Title,Genre,score
535,16 Blocks,"[Action, Crime, Drama]",3.0
375,2 Fast 2 Furious,"[Action, Crime, Thriller]",3.0
935,2 Guns,"[Action, Comedy, Crime]",3.0
856,21 Jump Street,"[Action, Comedy, Crime]",3.0
975,22 Jump Street,"[Action, Comedy, Crime]",3.0
938,A Good Day to Die Hard,"[Action, Crime, Thriller]",3.0
329,Austin Powers in Goldmember,"[Action, Comedy, Crime]",3.0
137,Bad Boys,"[Action, Comedy, Crime]",3.0
372,Bad Boys II,"[Action, Comedy, Crime]",3.0
8,Beverly Hills Cop,"[Action, Comedy, Crime]",3.0


,Title,Genre,score
375,2 Fast 2 Furious,"[Action, Crime, Thriller]",3.0
938,A Good Day to Die Hard,"[Action, Crime, Thriller]",3.0
651,Body of Lies,"[Action, Drama, Thriller]",3.0
507,Casino Royale,"[Action, Adventure, Thriller]",3.0
22,Commando,"[Action, Adventure, Thriller]",3.0
172,Con Air,"[Action, Crime, Thriller]",3.0
541,Crank,"[Action, Crime, Thriller]",3.0
382,Daredevil,"[Action, Crime, Thriller]",3.0
652,Death Race,"[Action, Sci-Fi, Thriller]",3.0
521,Deja Vu,"[Action, Sci-Fi, Thriller]",3.0


Second user weights: {'Action': 2, 'Thriller': 1}
Common movies in the two top-100 lists: 42%


## Offline learning

We start with offline learning. There are 2 steps: 
1. Collect the user's opinion on a few movies (e.g., 10)
2. Rank the other movies by logistic regression.

To keep the notebook fully reproducible, we simulate feedback from the latent user above: a movie is liked if its latent score is positive.

In [16]:
# Add a column to record the user's opinion (like / dislike)
movies = movies.assign(like=None)
feature_columns = [column for column in movies.columns if column != 'like']
true_user = user.iloc[0].reindex(feature_columns).astype(float)

def score_movies(weights, movie_ids=None):
    if movie_ids is None:
        movie_ids = movies.index
    scalar_input = np.isscalar(movie_ids)
    movie_ids = np.atleast_1d(movie_ids)
    X = movies.loc[movie_ids, feature_columns].to_numpy(dtype=float)
    scores = pd.Series(X @ np.asarray(weights, dtype=float), index=movie_ids, name='score')
    if scalar_input:
        return float(scores.iloc[0])
    return scores

def simulate_feedback(movie_ids):
    scores = score_movies(true_user, movie_ids)
    if np.isscalar(scores):
        return scores > 0
    return scores > 0

def fit_preference_model():
    observed = movies.like.notna()
    X_train = movies.loc[observed, feature_columns]
    y_train = movies.loc[observed, 'like'].astype(int)
    model = LogisticRegression(fit_intercept=False, random_state=0, solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)
    return model

def rank_movies(weights, movie_ids=None, score_name='score'):
    scores = score_movies(weights, movie_ids)
    if np.isscalar(scores):
        scores = pd.Series([scores], index=[movie_ids], name=score_name)
    else:
        scores = scores.rename(score_name)
    ranking = catalogue.loc[scores.index, ['Title', 'Genre']].copy()
    ranking[score_name] = scores
    return ranking.sort_values([score_name, 'Title'], ascending=[False, True])

In [17]:
# Select a random movie (not yet seen by the user)

def select_random_movie(rng=None):
    index = np.flatnonzero(movies.like.isna())
    if not len(index):
        index = np.arange(len(movies))
    if rng is None:
        return int(np.random.choice(index))
    return int(rng.choice(index))

In [18]:
# Create buttons

def create_expanded_button(description, button_style):
    return Button(
        description=description,
        button_style=button_style,
        layout=Layout())

def update_likes(button):
    global movie_id
    movies.loc[movie_id, 'like'] = button.description == 'like'
    
def update_poster():
    global movie_id
    img.value = catalogue.loc[movie_id].poster
    
def on_button_clicked(button):
    global movie_id
    update_likes(button)
    movie_id = select_random_movie()
    update_poster()    

In [28]:
# Setting the buttons
left_button = create_expanded_button('like', 'success')
right_button = create_expanded_button('dislike', 'danger')
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie(np.random.default_rng(0))
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button, 
    center=img,
    pane_widths=[0.3, 0.4, 0.3]
)

AppLayout(children=(Button(button_style='success', description='like', layout=Layout(grid_area='left-sidebar')…

## To do 2

* Give our opinion on some movies (e.g., 10), making sure that we get a few likes and a few dislikes.
* Apply logistic regression and display the other movies in order of preference (top movies first).
* Give our top-3 and bottom-3 genres, as predicted by the model.
* Here we simulate 12 feedbacks with a fixed random seed so that `Run all` completes end-to-end.

In [29]:
# Simulate offline feedback and display liked movies
movies = movies.assign(like=None)
offline_rng = np.random.default_rng(5)
offline_ids = offline_rng.choice(len(movies), size=12, replace=False)
movies.loc[offline_ids, 'like'] = simulate_feedback(offline_ids)

offline_feedback = rank_movies(true_user, offline_ids, score_name='latent_score')
offline_feedback['like'] = movies.loc[offline_feedback.index, 'like'].astype(bool)
display(offline_feedback[['Title', 'Genre', 'latent_score', 'like']])

likes = np.flatnonzero(movies.like == True)
display_posters(likes)

,Title,Genre,latent_score,like
1012,Furious 7,"[Action, Crime, Thriller]",3.0,True
650,Max Payne,"[Action, Crime, Drama]",3.0,True
831,The Mechanic,"[Action, Crime, Thriller]",3.0,True
482,Jarhead,"[Action, Drama, War]",2.0,True
397,Underworld,"[Action, Fantasy]",2.0,True
295,Hannibal,"[Crime, Drama, Thriller]",1.0,True
55,Dead Poets Society,"[Comedy, Drama]",0.0,False
688,He's Just Not That Into You,"[Comedy, Drama, Romance]",0.0,False
287,Shrek,"[Animation, Adventure, Comedy]",0.0,False
531,The Hills Have Eyes,[Horror],0.0,False


GridspecLayout(children=(Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xf…

In [30]:
# dislikes
dislikes = np.flatnonzero(movies.like == False)
display_posters(dislikes)

GridspecLayout(children=(Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xf…

In [31]:
model = fit_preference_model()

offline_genre_weights = pd.Series(model.coef_[0], index=feature_columns).sort_values(ascending=False)
unseen = np.flatnonzero(movies.like.isna())
offline_ranking = rank_movies(model.coef_[0], unseen, score_name='predicted_score')

display(offline_ranking.head(10))
display_posters(offline_ranking.head(10).index.to_numpy())

print('Top-3 genres:', offline_genre_weights.head(3).round(3).to_dict())
print('Bottom-3 genres:', offline_genre_weights.tail(3).round(3).to_dict())

,Title,Genre,predicted_score
375,2 Fast 2 Furious,"[Action, Crime, Thriller]",2.07524
938,A Good Day to Die Hard,"[Action, Crime, Thriller]",2.07524
172,Con Air,"[Action, Crime, Thriller]",2.07524
541,Crank,"[Action, Crime, Thriller]",2.07524
382,Daredevil,"[Action, Crime, Thriller]",2.07524
141,Desperado,"[Action, Crime, Thriller]",2.07524
780,Fast Five,"[Action, Crime, Thriller]",2.07524
435,Kill Bill: Vol. 2,"[Action, Crime, Thriller]",2.07524
832,Killer Elite,"[Action, Crime, Thriller]",2.07524
32,Lethal Weapon,"[Action, Crime, Thriller]",2.07524


Top-3 genres: {'Action': 0.913, 'Crime': 0.676, 'Thriller': 0.486}
Bottom-3 genres: {'Horror': -0.401, 'Sci-Fi': -0.5, 'Comedy': -0.852}


## Online learning

We now learn the user preferences online, as they come. For that, we use a Bayesian algorithm inspired by Thompson sampling. 

On each feedback provided by the user:
1. (Learning) The parameter (vector of weights) is learned.
2. (Sampling) A new parameter is sampled, assuming a Gaussian distribution.
3. (Action) The top movie for this new parameter, among movies not yet seen by the user, is proposed. 

Note that:
* In step 1, we retrain the estimator **from scratch**, using logistic regression on all training data samples (**no** online estimation).
* In step 2, we discard correlations (**diagonal** covariance matrix).

## To do 3

* Complete the function ``select_bayes`` below.
* Test it on some movies (e.g., 10), until we get a few likes and a few dislikes.
* Display the other movies in order of preference (top movies first).
* The last cell simulates 12 online interactions so that the notebook runs without manual clicks.

In [32]:
def select_bayes():
    observed = movies.like.notna()
    unseen = np.flatnonzero(movies.like.isna())
    seen_labels = set(movies.loc[observed, 'like'])

    if len(unseen) and len(seen_labels) == 2:
        model = fit_preference_model()
        X_seen = movies.loc[observed, feature_columns]
        p = model.predict_proba(X_seen)[:, 1]
        fisher_diag = np.sum((X_seen.to_numpy(dtype=float) ** 2) * (p * (1 - p))[:, None], axis=0) + 1e-6
        sampled_weights = bayes_rng.normal(loc=model.coef_[0], scale=1 / np.sqrt(fisher_diag))
        sampled_scores = movies.loc[unseen, feature_columns].to_numpy(dtype=float) @ sampled_weights
        return int(unseen[np.argmax(sampled_scores)])

    return select_random_movie(bayes_rng)

In [33]:
# reset
movies = movies.assign(like=None)
bayes_rng = np.random.default_rng(0)

In [34]:
def on_button_clicked(button):
    global movie_id
    update_likes(button)
    movie_id = select_bayes()
    update_poster()    

In [37]:
# Setting the buttons
left_button = create_expanded_button('like', 'success')
right_button = create_expanded_button('dislike', 'danger')
left_button.on_click(on_button_clicked)
right_button.on_click(on_button_clicked)

# Setting the movie poster
movie_id = select_random_movie(np.random.default_rng(1))
img = get_poster(movie_id, scale=1.5)

# Display
AppLayout(
    left_sidebar=left_button,
    right_sidebar=right_button, 
    center=img,
    pane_widths=[0.3, 0.4, 0.3]
)

AppLayout(children=(Button(button_style='success', description='like', layout=Layout(grid_area='left-sidebar')…

In [ ]:


# Simulate 12 online interactions and display the final ranking
history = []

for step in range(12):
    movie_id = select_bayes()
    like = bool(simulate_feedback(movie_id))
    movies.loc[movie_id, 'like'] = like
    history.append({
        'step': step + 1,
        'Title': catalogue.loc[movie_id, 'Title'],
        'Genre': catalogue.loc[movie_id, 'Genre'],
        'latent_score': score_movies(true_user, movie_id),
        'like': like,
    })

online_feedback = pd.DataFrame(history)
display(online_feedback)

online_likes = np.flatnonzero(movies.like == True)
online_dislikes = np.flatnonzero(movies.like == False)
print(f'Collected {len(online_likes)} likes and {len(online_dislikes)} dislikes.')

display_posters(online_likes)

online_model = fit_preference_model()
online_ranking = rank_movies(online_model.coef_[0], np.flatnonzero(movies.like.isna()), score_name='predicted_score')
display(online_ranking.head(10))
display_posters(online_ranking.head(10).index.to_numpy())

,step,Title,Genre,latent_score,like
0,1,The Iron Giant,"[Animation, Action, Adventure]",2.0,True
1,2,Hellboy,"[Action, Fantasy, Horror]",2.0,True
2,3,Legends of the Fall,"[Drama, Romance, War]",0.0,False
3,4,JFK,"[Drama, History, Thriller]",0.0,False
4,5,Fahrenheit 9/11,"[Documentary, Comedy, Drama]",0.0,False
5,6,The Tourist,"[Action, Romance, Thriller]",2.0,True
6,7,Alexander,"[Action, Adventure, Biography]",2.0,True
7,8,Mamma Mia!,"[Comedy, Musical, Romance]",0.0,False
8,9,Jumanji,"[Adventure, Family, Fantasy]",0.0,False
9,10,Wild Wild West,"[Action, Western, Comedy]",2.0,True


Collected 13 likes and 23 dislikes.


,Title,Genre,predicted_score
45,Die Hard,"[Action, Thriller]",1.774985
998,John Wick,"[Action, Thriller]",1.774985
929,Olympus Has Fallen,"[Action, Thriller]",1.774985
394,Once Upon a Time in Mexico,"[Action, Thriller]",1.774985
539,Snakes on a Plane,"[Action, Thriller]",1.774985
411,The Bourne Supremacy,"[Action, Thriller]",1.774985
555,The Bourne Ultimatum,"[Action, Thriller]",1.774985
840,The Dark Knight Rises,"[Action, Thriller]",1.774985
750,Unstoppable,"[Action, Thriller]",1.774985
507,Casino Royale,"[Action, Adventure, Thriller]",1.725622


GridspecLayout(children=(Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xf…